# 01 — Download the GOES flare catalog

Builds the input data file `data/gannon_storm_2024_flares.csv` from the Heliophysics Event Knowledgebase (HEK).

* Time window: **2024-04-28 → 2024-05-15** (the days when AR 13663 and AR 13664 were over the visible solar disk).
* Event type: GOES X-ray flares of class **C1.0** or above.
* Output columns: `event_starttime`, `event_peaktime`, `event_endtime`, `fl_goescls`, `ar_noaanum`.

The CSV produced by this notebook is the starting point for **Table 1** of the paper and for notebooks `03`, `05`, `06`, and `07`.


In [ ]:
from pathlib import Path
from sunpy.net import Fido, attrs as a

DATA_DIR = Path('..').resolve() / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = DATA_DIR / 'gannon_storm_2024_flares.csv'


In [ ]:
# Query the HEK for all GOES C-class or stronger flares in the window
tstart, tend = '2024/04/28', '2024/05/15'

result = Fido.search(
    a.Time(tstart, tend),
    a.hek.EventType('FL'),
    a.hek.FL.GOESCls > 'C1.0',
    a.hek.OBS.Observatory == 'GOES',
)
result


In [ ]:
# Keep only the columns we need for the paper
hek = result['hek']
filtered = hek['event_starttime', 'event_peaktime',
              'event_endtime', 'fl_goescls', 'ar_noaanum']
filtered


In [ ]:
# Sanity-check: sort by magnitude (X > M > C, then numeric suffix)
by_magnitude = sorted(
    filtered,
    key=lambda r: ord(r['fl_goescls'][0]) + float(r['fl_goescls'][1:]),
    reverse=True,
)
for flare in by_magnitude[:10]:
    print(f"Class {flare['fl_goescls']} on {flare['event_starttime']} (AR {flare['ar_noaanum']})")


In [ ]:
# Write to disk for the downstream notebooks to consume
filtered.write(OUTPUT_CSV, format='csv', overwrite=True)
print(f'Wrote {OUTPUT_CSV} ({len(filtered)} rows)')
